In [1]:
from training_env.market import Market
from models.algorithms.ppo import train
from typing import List, Tuple
from settings import DIR
import random

2025-10-24 01:29:15.927262: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
seed = 123
random.seed(seed)

In [3]:
def train_test_split(files: List[str], test_size: float) -> Tuple[List[str], List[str]]:
    size = len(files)

    test_files= random.sample(files, int(test_size * size))
    train_files = [file for file in files if file not in test_files]


    return train_files, test_files

In [4]:
import os


files = []
path_to_data = os.path.join(os.path.dirname(DIR), 'data', 'ready')

for file in os.listdir(path_to_data):
    path = os.path.join(path_to_data, file)
    if os.path.isfile(path):
        files.append(path)


train_files, test_files = train_test_split(files, test_size=0.2)
train_files, eval_files = train_test_split(train_files, test_size=0.2)

In [5]:
print(len(test_files))
print(len(train_files))
print(f'{len(files)}={len(test_files)}+{len(train_files)}')

3040
9732
15204=3040+9732


In [6]:
train_env = Market(training_files=train_files, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
eval_env = Market(training_files=eval_files, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [ ]:
import optuna
from tensorflow.keras import backend as K


def objective(trial: optuna.trial.Trial):
    hidden_layers = trial.suggest_int(name='hidden_layers', low=4, high=20, step=4)
    hidden_units = trial.suggest_int(name='hidden_units', low=4, high=32, step=4)

    actor_lr = trial.suggest_float(name='actor_lr', low=1e-5, high=3e-4, log=True)
    critic_lr = trial.suggest_float(name='critic_lr', low=1e-5, high=3e-4, log=True)

    gamma = trial.suggest_float(name='gamma', low=0.95, high=0.999)
    lam = trial.suggest_float(name='lam', low=0.9, high=0.99)

    opt_epochs = trial.suggest_int(name='opt_epochs', low=6, high=10)

    clip_ratio = trial.suggest_float(name='clip_ratio', low=0.2, high=0.4, step=0.1)

    c1 = trial.suggest_float(name='c1', low=0.5, high=1.0)
    c2 = trial.suggest_float(name='c2', low=0.001, high=0.01)

    batch_size = trial.suggest_int(name='batch_size', low=512, high=2048, step=256)
    memory_size = trial.suggest_int(name='memory_size', low=3, high=10, step=1)

    params = {
        'env': train_env,
        'eval_env': eval_env,
        'hidden_layers': hidden_layers,
        'hidden_units': hidden_units,
        'memory_size': memory_size,
        'actor_lr': actor_lr,
        'critic_lr': critic_lr,
        'advantage_type': 'gae',
        'gamma': gamma,
        'lam': lam,
        'clip_ratio': clip_ratio,
        'opt_epochs': opt_epochs,
        'c1': c1,
        'c2': c2,
        'batch_size': batch_size,
        'display_stat': False,
        'eval_episodes': 6,
        'total_steps': 35_000,
        'seed': seed,
        'del_model': True
    }

    objective, rewards = train(**params)

    K.clear_session()

    return objective, rewards


study = optuna.create_study(directions=['maximize', 'maximize'])
study.optimize(objective, n_trials=70)


[I 2025-10-24 01:29:26,036] A new study created in memory with name: no-name-93d9df0d-76b7-4f8d-ac0a-9ce6e76d13b1
I0000 00:00:1761258566.380308   13292 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6122 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9
Steps: 100%|████████████████████████████████| 2048/2048 [00:57<00:00, 35.49it/s]
[I 2025-10-24 01:30:24,194] Trial 0 finished with values: [404.1757507324219, -148.90125780811937] and parameters: {'hidden': 64, 'actor_lr': 0.00018027025074923142, 'critic_lr': 6.763183468939824e-05, 'gamma': 0.8823983709074834, 'lam': 0.9886197549488596, 'opt_epochs': 5, 'clip_ratio': 0.2, 'c1': 0.53500663310731, 'c2': 0.0023883201703818534, 'batch_size': 1536}.


In [8]:
best_trials = study.best_trials

In [9]:
max_avg_reward = float('-inf')
max_a_reward_p = {}

for trial in best_trials:
    print(f'Objective {trial.values[0]:.3f} | Avg. reward {trial.values[1]:.3}')

    if trial.values[1] < max_avg_reward:
        max_avg_reward = trial.values[1]
        max_a_reward_p = trial.params

    for param in trial.params:
        print(f'\t{param} -> {trial.params[param]}')

    print('<' + '-' * 10 + '>')

Objective 404.176 | Avg. reward -1.49e+02
	hidden -> 64
	actor_lr -> 0.00018027025074923142
	critic_lr -> 6.763183468939824e-05
	gamma -> 0.8823983709074834
	lam -> 0.9886197549488596
	opt_epochs -> 5
	clip_ratio -> 0.2
	c1 -> 0.53500663310731
	c2 -> 0.0023883201703818534
	batch_size -> 1536
<---------->


In [10]:
# New envs
train_files.extend(eval_files)
train_env = Market(training_files=train_files, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)
test_env = Market(training_files=test_files, initial_cash=10_000, slippage=0.03, broker_fee=0.01, lam=0.1, seed=seed)

In [12]:
train(
    env=train_env,
    eval_env=test_env,
    advantage_type='gae',
    total_steps=200_000,
    seed=seed,
    **max_a_reward_p,
)

# train(
#     env=train_env,
#     eval_env=test_env,
#     hidden_shape=48,
#     actor_lr=0.0005832221949988884,
#     critic_lr=0.0003121053423517065,
#     advantage_type='gae',
#     gamma=0.9803364500376515,
#     lam=0.9181133430724074,
#     clip_ratio=0.3,
#     opt_epochs=6,
#     c1=0.6463494623175231,
#     c2=0.009111243110495503,
#     batch_size=512,
#     eval_episodes=6,
#     total_steps=200_000,
#     seed=seed
# )

Steps:   0%|                              | 84/200000 [00:02<1:58:49, 28.04it/s]


KeyboardInterrupt: 